### CONEXION DDBB OLIST

In [1]:
%pip install PyMySQL
from sqlalchemy import create_engine, text
import ssl

## CONEXION BBDD MYSQL ##
DB_USER = "nuclio"
DB_PASS = "nuclioTFM6"
DB_HOST = "nuclio.mysql.database.azure.com"
DB_NAME = "olist"

# Crear engine apuntando a la base 'olist'
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:3306/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True,
    connect_args={"ssl": {"cert_reqs": ssl.CERT_NONE, "check_hostname": False}} 
)

# tablas 'olist'
with engine.connect() as conn:
    tables = conn.execute(text("SHOW TABLES")).fetchall()
    tables = [row[0] for row in tables]   # convertir a lista simple de strings
    
    print("Tablas en la base 'olist':")
    for t in tables:
        print("-", t)



Note: you may need to restart the kernel to use updated packages.
Tablas en la base 'olist':
- dash_olist_categorias_resumen
- dash_olist_demorados
- dash_olist_sellers
- dash_olist_states
- dash_olist_ventas_meses
- dash_sentiment_analysis
- distribucion_pedidos
- olist_customers_dataset
- olist_geolocation_dataset
- olist_order_items_dataset
- olist_order_payments_dataset
- olist_order_reviews_dataset
- olist_orders_dataset
- olist_products_dataset
- olist_sellers_dataset
- pedidos_por_tiempo
- product_category_name_translation


In [2]:
import pandas as pd
from IPython.display import display, Markdown

# --- Cargar datos ---
query = """
SELECT 
    pct.product_category_name_english AS product_category_name,
    i.price,
    i.seller_id,
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp,
    r.review_score
FROM olist_order_items_dataset i
LEFT JOIN olist_products_dataset p
    ON i.product_id = p.product_id
LEFT JOIN product_category_name_translation pct
    ON p.product_category_name = pct.product_category_name
LEFT JOIN olist_orders_dataset o
    ON i.order_id = o.order_id
LEFT JOIN olist_customers_dataset c
    ON o.customer_id = c.customer_id
LEFT JOIN olist_order_reviews_dataset r
    ON o.order_id = r.order_id
WHERE o.order_status <> 'canceled'
"""
df = pd.read_sql_query(query, con=engine)

# --- Procesamiento temporal ---
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")
df["order_year"] = df["order_purchase_timestamp"].dt.year
df = df[df["order_year"].isin([2017, 2018])]

# ---  Métricas por categoría y año ---
group_cols = ["product_category_name", "order_year"]

agg_df = (
    df.groupby(group_cols, dropna=False)
      .agg(
          TotalSales=("price", "sum"),
          OrdersQty=("order_id", "nunique"),
          UniqueSellers=("seller_id", "nunique"),
          Customers=("customer_unique_id", "nunique"),
          avg_score=("review_score", "mean")
      )
      .reset_index()
)

# ---  Clientes recurrentes ---
repeat_customers = (
    df.groupby(group_cols + ["customer_unique_id"])
      .agg(order_count=("order_id", "nunique"))
      .reset_index()
)
repeat_customers = (
    repeat_customers[repeat_customers["order_count"] > 1]
    .groupby(group_cols)
    .agg(RepeatCustomers=("customer_unique_id", "nunique"))
    .reset_index()
)

# ---  Merge de métricas ---
merged_detailed = agg_df.merge(repeat_customers, on=group_cols, how="left").fillna({"RepeatCustomers": 0})

# ---  Ticket medio ---
merged_detailed["AvgTicket"] = (merged_detailed["TotalSales"] / merged_detailed["OrdersQty"]).round(2)

# ---  Redondear y limpiar ---
merged_detailed["avg_score"] = merged_detailed["avg_score"].round(2)

# ---  Mostrar resultados ---
display(Markdown("#### Ventas y métricas detalladas por categoría y año (2017–2018)"))
display(
    merged_detailed[
        [
            "product_category_name",
            "order_year",
            "TotalSales",
            "OrdersQty",
            "UniqueSellers",
            "Customers",
            "avg_score",
            "RepeatCustomers",
            "AvgTicket"
        ]
    ].sort_values(by=["order_year", "TotalSales"], ascending=[True, False]).head(20)
)

# ---  Totales de vendedores únicos reales por año ---
unique_sellers_by_year = (
    df.groupby("order_year")["seller_id"]
    .nunique()
    .reset_index(name="UniqueSellers")
    .sort_values("order_year")
)

total_unique_sellers_all_years = df["seller_id"].nunique()

display(Markdown("### Total de vendedores únicos por año (sin duplicidad por categoría)"))
display(unique_sellers_by_year)

display(Markdown(f"### Total de vendedores únicos en todos los años: **{total_unique_sellers_all_years:,}**"))


#### Ventas y métricas detalladas por categoría y año (2017–2018)

,product_category_name,order_year,TotalSales,OrdersQty,UniqueSellers,Customers,avg_score,RepeatCustomers,AvgTicket
14,bed_bath_table,2017,497970.94,4496,104,4346,3.87,139.0,110.76
139,watches_gifts,2017,486519.02,2114,58,2095,3.33,18.0,230.14
86,health_beauty,2017,481142.73,3382,236,3297,4.00,82.0,142.27
129,sports_leisure,2017,447546.59,3621,285,3517,2.80,95.0,123.60
30,computers_accessories,2017,400490.61,2606,176,2567,5.00,37.0,153.68
40,cool_stuff,2017,388971.98,2200,183,2195,2.40,5.0,176.81
78,furniture_decor,2017,336272.26,3187,246,3113,5.00,72.0,105.51
137,toys,2017,305991.37,2438,136,2417,4.75,20.0,125.51
84,garden_tools,2017,265131.73,1969,150,1952,4.73,17.0,134.65
10,auto,2017,238545.04,1415,179,1398,5.00,17.0,168.58


### Total de vendedores únicos por año (sin duplicidad por categoría)

,order_year,UniqueSellers
0,2017,1766
1,2018,2354


### Total de vendedores únicos en todos los años: **3,029**

In [3]:
# total de ventas por año
check_totals = (
    merged_detailed.groupby("order_year")["TotalSales"].sum().reset_index()
)
display(check_totals)


,order_year,TotalSales
0,2017,6108492.27
1,2018,7341182.41


In [4]:
# --- Total de vendedores únicos por año (sin duplicidad por categoría) ---
unique_sellers_total = (
    df.groupby("order_year")["seller_id"]
      .nunique()
      .reset_index(name="UniqueSellers")
      .sort_values("order_year")
)

# ---  Total de vendedores únicos en todos los años ---
total_unique_sellers_all_years = df["seller_id"].nunique()

display(Markdown("### Total de vendedores únicos por año (reales)"))
display(unique_sellers_total)

display(Markdown(f"### Total de vendedores únicos en todos los años: **{total_unique_sellers_all_years:,}**"))



### Total de vendedores únicos por año (reales)

,order_year,UniqueSellers
0,2017,1766
1,2018,2354


### Total de vendedores únicos en todos los años: **3,029**

In [ ]:
from sqlalchemy import text
from sqlalchemy.types import Float, Integer, String

# ---  Guardar DataFrame en la base de datos ---
table_name = "Dash_olist_categorias_resumen"  # nombre estandarizado (mayúscula inicial)

# ---  Eliminar tabla si existe (para evitar conflictos por reflexión) ---
with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS `{table_name}`;"))

# ---  Definir tipos de columnas ---
dtype_map = {
    "product_category_name": String(100),
    "order_year": Integer(),
    "TotalSales": Float(),
    "OrdersQty": Integer(),
    "UniqueSellers": Integer(),
    "Customers": Integer(),
    "avg_score": Float(),
    "RepeatCustomers": Integer(),
    "AvgTicket": Float(),
}

# ---  Crear y cargar tabla ---
merged_detailed.to_sql(
    name=table_name,
    con=engine,
    if_exists="fail",   
    index=False,
    dtype=dtype_map,
    method="multi"
)

print(f" Tabla '{table_name}' creada y cargada correctamente en la base de datos.")

# ---  Validación opcional: mostrar número de registros cargados ---
with engine.connect() as conn:
    result = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`;"))
    total_rows = result.scalar()
    print(f" Total de registros insertados: {total_rows:,}")


C:\Users\HY269\AppData\Local\Temp\ipykernel_13864\2111715177.py:25: UserWarning: The provided table name 'Dash_olist_categorias_resumen' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  merged_detailed.to_sql(


 Tabla 'Dash_olist_categorias_resumen' creada y cargada correctamente en la base de datos.
 Total de registros insertados: 143
